 Imports and configuration

In [ ]:
import os
import random
from typing import Dict, List, Tuple

import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import (
	accuracy_score,
	classification_report,
	confusion_matrix,
	precision_recall_curve,
	roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow import keras
from tensorflow.keras import layers


RANDOM_SEED = 42
IMG_HEIGHT = 166
IMG_WIDTH = 400
IMG_CHANNELS = 3

N_SPLITS = 5  # stratified k-fold


def set_global_seed(seed: int = RANDOM_SEED) -> None:
	random.seed(seed)
	np.random.seed(seed)
	keras.utils.set_random_seed(seed)


set_global_seed(RANDOM_SEED)



 Data loading

In [ ]:
def load_images_from_folder(folder: str, label: int) -> Tuple[np.ndarray, np.ndarray]:
	images: List[np.ndarray] = []
	labels: List[int] = []
	for fname in os.listdir(folder):
		fpath = os.path.join(folder, fname)
		if not os.path.isfile(fpath):
			continue
		img = cv2.imread(fpath)
		if img is None:
			continue
		img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
		img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
		images.append(img)
		labels.append(label)
	return np.array(images, dtype=np.uint8), np.array(labels, dtype=np.int32)


def load_dataset(base_dir: str = "Data-SalmonScan") -> Tuple[np.ndarray, np.ndarray]:
	fresh_dir = os.path.join(base_dir, "FreshFish")
	infected_dir = os.path.join(base_dir, "InfectedFish")

	fresh_images, fresh_labels = load_images_from_folder(fresh_dir, label=0)
	infected_images, infected_labels = load_images_from_folder(infected_dir, label=1)

	X = np.concatenate([fresh_images, infected_images], axis=0)
	y = np.concatenate([fresh_labels, infected_labels], axis=0)

	return X, y


X_raw, y_raw = load_dataset()



 Preprocessing (light CLAHE + normalization)

In [ ]:
def apply_clahe_rgb(image: np.ndarray, clip_limit: float = 2.0, tile_grid_size: Tuple[int, int] = (8, 8)) -> np.ndarray:
	lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
	l, a, b = cv2.split(lab)
	clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
	l_clahe = clahe.apply(l)
	lab_clahe = cv2.merge((l_clahe, a, b))
	rgb_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
	return rgb_clahe


def preprocess_images(images: np.ndarray, use_clahe: bool = True) -> np.ndarray:
	processed = []
	for img in images:
		if use_clahe:
			img = apply_clahe_rgb(img)
		img = img.astype("float32") / 255.0
		processed.append(img)
	return np.stack(processed, axis=0)


X_all = preprocess_images(X_raw, use_clahe=True)



 Recommended train/test split

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
	X_all,
	y_raw,
	test_size=0.2,
	stratify=y_raw,
	random_state=RANDOM_SEED,
)



 Data generators with on-the-fly augmentation (train only)

In [ ]:
def create_image_data_generator() -> keras.preprocessing.image.ImageDataGenerator:
	return keras.preprocessing.image.ImageDataGenerator(
		rotation_range=25,
		width_shift_range=0.1,
		height_shift_range=0.1,
		zoom_range=0.15,
		horizontal_flip=True,
		vertical_flip=False,
		fill_mode="nearest",
	)


def create_train_generator(
	X: np.ndarray,
	y: np.ndarray,
	batch_size: int = 16,
) -> keras.preprocessing.image.NumpyArrayIterator:
	datagen = create_image_data_generator()
	return datagen.flow(X, y, batch_size=batch_size, shuffle=True)



 Model definitions (1, 2, 3 conv+pool blocks)

In [ ]:
def build_cnn_model(num_blocks: int, input_shape: Tuple[int, int, int] = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)) -> keras.Model:
	inputs = keras.Input(shape=input_shape)

	x = inputs
	filters = 32
	for _ in range(num_blocks):
		x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
		x = layers.BatchNormalization()(x)
		x = layers.MaxPooling2D((2, 2))(x)
		x = layers.Dropout(0.25)(x)
		filters *= 2

	x = layers.Flatten()(x)
	x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
	x = layers.BatchNormalization()(x)
	x = layers.Dropout(0.5)(x)
	outputs = layers.Dense(1, activation="sigmoid")(x)

	model = keras.Model(inputs=inputs, outputs=outputs, name=f"cnn_{num_blocks}_blocks")
	model.compile(
		optimizer=keras.optimizers.Adam(learning_rate=1e-3),
		loss="binary_crossentropy",
		metrics=["accuracy"],
	)
	return model


models_config = {
	"cnn_1_block": 1,
	"cnn_2_blocks": 2,
	"cnn_3_blocks": 3,
}



 Stratified k-fold cross-validation (on train_full)

In [ ]:
def train_with_stratified_kfold(
	X: np.ndarray,
	y: np.ndarray,
	model_blocks: int,
	n_splits: int = N_SPLITS,
	epochs: int = 40,
	batch_size: int = 16,
) -> pd.DataFrame:
	skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
	fold_metrics: List[Dict[str, float]] = []

	for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
		X_train, X_val = X[train_idx], X[val_idx]
		y_train, y_val = y[train_idx], y[val_idx]

		train_gen = create_train_generator(X_train, y_train, batch_size=batch_size)

		model = build_cnn_model(num_blocks=model_blocks)

		steps_per_epoch = max(1, len(X_train) // batch_size)

		callbacks = [
			keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
			keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
		]

		history = model.fit(
			train_gen,
			steps_per_epoch=steps_per_epoch,
			epochs=epochs,
			validation_data=(X_val, y_val),
			callbacks=callbacks,
			verbose=0,
		)

		val_pred_proba = model.predict(X_val, verbose=0).ravel()
		val_pred = (val_pred_proba >= 0.5).astype(int)

		acc = accuracy_score(y_val, val_pred)
		auc = roc_auc_score(y_val, val_pred_proba)

		cm = confusion_matrix(y_val, val_pred)
		tn, fp, fn, tp = cm.ravel()
		sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
		specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

		fold_metrics.append(
			{
				"fold": fold,
				"val_accuracy": acc,
				"val_auc": auc,
				"val_sensitivity": sensitivity,
				"val_specificity": specificity,
			}
		)

	return pd.DataFrame(fold_metrics)


cv_results: Dict[str, pd.DataFrame] = {}
for model_name, blocks in models_config.items():
	cv_results[model_name] = train_with_stratified_kfold(X_train_full, y_train_full, model_blocks=blocks)



 Aggregate CV results into a single DataFrame

In [ ]:
cv_summary_rows: List[Dict[str, float]] = []
for model_name, df in cv_results.items():
	row = {
		"model": model_name,
		"folds": len(df),
		"mean_val_accuracy": df["val_accuracy"].mean(),
		"std_val_accuracy": df["val_accuracy"].std(),
		"mean_val_auc": df["val_auc"].mean(),
		"mean_val_sensitivity": df["val_sensitivity"].mean(),
		"mean_val_specificity": df["val_specificity"].mean(),
	}
	cv_summary_rows.append(row)

cv_summary_df = pd.DataFrame(cv_summary_rows)



 Train final models on full train set and evaluate on held-out test set

In [ ]:
def train_final_model(
	X_train: np.ndarray,
	y_train: np.ndarray,
	blocks: int,
	epochs: int = 40,
	batch_size: int = 16,
) -> keras.Model:
	train_gen = create_train_generator(X_train, y_train, batch_size=batch_size)
	model = build_cnn_model(num_blocks=blocks)
	steps_per_epoch = max(1, len(X_train) // batch_size)

	callbacks = [
		keras.callbacks.EarlyStopping(monitor="loss", patience=6, restore_best_weights=True),
		keras.callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5, patience=3, min_lr=1e-5),
	]

	model.fit(
		train_gen,
		steps_per_epoch=steps_per_epoch,
		epochs=epochs,
		callbacks=callbacks,
		verbose=0,
	)
	return model


final_models: Dict[str, keras.Model] = {}
test_eval_rows: List[Dict[str, float]] = []

for model_name, blocks in models_config.items():
	model = train_final_model(X_train_full, y_train_full, blocks=blocks)
	final_models[model_name] = model

	y_proba = model.predict(X_test, verbose=0).ravel()
	y_pred = (y_proba >= 0.5).astype(int)

	acc = accuracy_score(y_test, y_pred)
	auc = roc_auc_score(y_test, y_proba)
	cm = confusion_matrix(y_test, y_pred)
	tn, fp, fn, tp = cm.ravel()
	sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
	specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

	test_eval_rows.append(
		{
			"model": model_name,
			"test_accuracy": acc,
			"test_auc": auc,
			"test_sensitivity": sensitivity,
			"test_specificity": specificity,
			"tp": tp,
			"fp": fp,
			"fn": fn,
			"tn": tn,
		}
	)

test_eval_df = pd.DataFrame(test_eval_rows)



 Threshold tuning and optional human review regime

In [ ]:
def threshold_and_human_review_stats(
	y_true: np.ndarray,
	y_proba: np.ndarray,
	threshold: float,
	review_band: Tuple[float, float] = (0.4, 0.6),
) -> Dict[str, float]:
	lower, upper = review_band
	uncertain_mask = (y_proba >= lower) & (y_proba <= upper)
	auto_mask = ~uncertain_mask

	y_pred_auto = (y_proba[auto_mask] >= threshold).astype(int)
	y_true_auto = y_true[auto_mask]

	if len(y_true_auto) > 0:
		cm_auto = confusion_matrix(y_true_auto, y_pred_auto)
		tn_a, fp_a, fn_a, tp_a = cm_auto.ravel()
		sensitivity_auto = tp_a / (tp_a + fn_a) if (tp_a + fn_a) > 0 else 0.0
		specificity_auto = tn_a / (tn_a + fp_a) if (tn_a + fp_a) > 0 else 0.0
		acc_auto = accuracy_score(y_true_auto, y_pred_auto)
	else:
		sensitivity_auto = 0.0
		specificity_auto = 0.0
		acc_auto = 0.0

	auto_fraction = len(y_true_auto) / len(y_true) if len(y_true) > 0 else 0.0

	return {
		"threshold": threshold,
		"review_lower": lower,
		"review_upper": upper,
		"auto_fraction": auto_fraction,
		"auto_accuracy": acc_auto,
		"auto_sensitivity": sensitivity_auto,
		"auto_specificity": specificity_auto,
	}


threshold_analysis_rows: List[Dict[str, float]] = []

for model_name, model in final_models.items():
	y_proba = model.predict(X_test, verbose=0).ravel()

	thresholds = [0.4, 0.5, 0.6]
	for thr in thresholds:
		stats = threshold_and_human_review_stats(y_test, y_proba, threshold=thr)
		stats["model"] = model_name
		threshold_analysis_rows.append(stats)

threshold_analysis_df = pd.DataFrame(threshold_analysis_rows)



 Error and case analysis for best model (by test_sensitivity, then test_auc)

In [ ]:
best_model_row = test_eval_df.sort_values([
	"test_sensitivity",
	"test_auc",
], ascending=False).iloc[0]

best_model_name = best_model_row["model"]
best_model = final_models[best_model_name]

y_proba_best = best_model.predict(X_test, verbose=0).ravel()
y_pred_best = (y_proba_best >= 0.5).astype(int)

cm_best = confusion_matrix(y_test, y_pred_best)
tn_b, fp_b, fn_b, tp_b = cm_best.ravel()

false_negative_indices = np.where((y_test == 1) & (y_pred_best == 0))[0]
false_positive_indices = np.where((y_test == 0) & (y_pred_best == 1))[0]

error_analysis_df = pd.DataFrame(
	{
		"index": np.concatenate([false_negative_indices, false_positive_indices]),
		"true_label": np.concatenate([np.ones_like(false_negative_indices), np.zeros_like(false_positive_indices)]),
		"pred_label": np.concatenate([np.zeros_like(false_negative_indices), np.ones_like(false_positive_indices)]),
		"pred_proba": np.concatenate([
			y_proba_best[false_negative_indices],
			y_proba_best[false_positive_indices],
		]),
	}
)



 Precision-recall and Six Sigma oriented statistics for each final model

In [ ]:
six_sigma_rows: List[Dict[str, float]] = []

for model_name, model in final_models.items():
	y_proba = model.predict(X_test, verbose=0).ravel()
	y_pred = (y_proba >= 0.5).astype(int)

	precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba)
	auc_score = roc_auc_score(y_test, y_proba)
	cm = confusion_matrix(y_test, y_pred)
	tn, fp, fn, tp = cm.ravel()

	total = tn + fp + fn + tp
	defect_rate = (fn + fp) / total if total > 0 else 0.0

	six_sigma_rows.append(
		{
			"model": model_name,
			"auc": auc_score,
			"accuracy": accuracy_score(y_test, y_pred),
			"sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
			"specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
			"defect_rate": defect_rate,
			"false_positives": fp,
			"false_negatives": fn,
		}
	)

six_sigma_df = pd.DataFrame(six_sigma_rows)



 Key DataFrames available for further analysis or export:

 - cv_summary_df: mean and std of k-fold validation metrics per model

 - test_eval_df: performance on held-out test set

 - threshold_analysis_df: auto vs review statistics for different thresholds

 - error_analysis_df: misclassified test cases for best model

 - six_sigma_df: summary metrics oriented toward quality / Six Sigma analysis